## Setup and Imports

In [ ]:
# Standard Library Imports
import sys
import os
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Add src directory to path
project_root = Path().resolve().parents[1]
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    
# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Project utilities
from utils.helpers import load_orders, set_style, save_figure

set_style()

print("All imports successful!")
print(f"Project root: {project_root}")
print(f'pandas version: {pd.__version__}')
print(f'numpy version: {np.__version__}')

## Load Data

In [ ]:
processed_path = project_root / 'data' / 'processed'
input_file = processed_path / 'orders_with_anomaly_flag.csv'

df_raw = pd.read_csv(input_file)
df = df_raw.copy()

# Parse timestamps
df['PURCHASE_TS'] = pd.to_datetime(df['PURCHASE_TS'], errors='coerce')
df['SHIP_TS'] = pd.to_datetime(df['SHIP_TS'], errors='coerce')

# Extract date components
df['PURCHASE_YEAR'] = df['PURCHASE_TS'].dt.year
df['PURCHASE_MONTH'] = df['PURCHASE_TS'].dt.month
df['PURCHASE_YEARMONTH'] = df['PURCHASE_TS'].dt.to_period('M')

print(f'Loaded: {len(df):,} rows, {len(df.columns)} columns')
print(f'Columns: {list(df.columns)}')
print(f'\nIS_ANOMALY breakdown:\n{df["IS_ANOMALY"].value_counts()}')

## Overall Price Distribution

In [ ]:
# Basic price statistics across dataset

print('Overall price statistics')
price_stats = df['USD_PRICE'].describe()
print(f" Count:   {price_stats['count']:>10,.0f}")
print(f" Mean:    {price_stats['mean']:>10,.2f}")
print(f" Std Dev: {price_stats['std']:>10,.2f}")
print(f" Min:     {price_stats['min']:>10,.2f}")
print(f" 25%:     {price_stats['25%']:>10,.2f}")
print(f" Median:  {price_stats['50%']:>10,.2f}")
print(f" 75%:     {price_stats['75%']:>10,.2f}")
print(f" Max:     {price_stats['max']:>10,.2f}")
print()

# Check for skewness
# A large gap between mean and median suggests skewness, which is common in price data due to outliers.
# If the mean is significantly higher than the median, it indicates a right-skewed distribution, often caused by a few very high-priced orders.
# Conversely, if the mean is lower than the median, it suggests a left-skewed distribution, which is less common in price data.
mean = price_stats['mean']
median = price_stats['50%']
gap = mean - median
print(f"Mean vs Median gap: ${gap:.2f}")
if gap > 0:
    print(f"The mean is higher than the median, indicating a right-skewed distribution likely due to high-priced outliers.")
elif gap < 0:
    print(f"The median is higher than the mean, indicating a left-skewed distribution, which is less common in price data.")
else:
    print(f"The mean and median are equal, suggesting a symmetric distribution, which is unusual for price data and may indicate data issues.")

In [ ]:

print(f"Price Points of Interest:")
zero_prices = df[df['USD_PRICE'] == 0]
null_prices = df[df['USD_PRICE'].isnull()]
under_10    = df[df['USD_PRICE'] < 10]
over_1000   = df[df['USD_PRICE'] > 1000]
over_2000   = df[df['USD_PRICE'] > 2000]

print(f" Exactly $0:          {len(zero_prices):>6,} orders ({len(zero_prices)/len(df)*100:.2f}%)")
print(f" Null/missing prices: {len(null_prices):>6,} orders ({len(null_prices)/len(df)*100:.2f}%)")
print(f" Under $10:           {len(under_10):>6,} orders ({len(under_10)/len(df)*100:.2f}%)")
print(f" Over $1,000:         {len(over_1000):>6,} orders ({len(over_1000)/len(df)*100:.2f}%)")
print(f" Over $2,000:         {len(over_2000):>6,} orders ({len(over_2000)/len(df)*100:.2f}%)")

## Extreme Value Deep Dive

In [ ]:
# Calculate IQR and outlier thresholds for dataset
Q1 = df['USD_PRICE'].quantile(0.25) # 25th percentile
Q3 = df['USD_PRICE'].quantile(0.75) # 75th percentile
IQR = Q3 - Q1 # Interquartile Range

lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

print(f"Price outlier thresholds:")
print(f' Q1 (25th percentile): ${Q1:.2f}')
print(f' Q3 (75th percentile): ${Q3:.2f}')
print(f' IQR:                  ${IQR:.2f}')
print(f' Lower bound:          ${lower_bound:.2f}')
print(f' Upper bound:          ${upper_bound:.2f}')
print()

low_outliers = df[df['USD_PRICE'] < lower_bound]
high_outliers = df[df['USD_PRICE'] > upper_bound]

print(f' Low outliers  (< ${lower_bound:.2f}): {len(low_outliers):,}')
print(f' High outliers (> ${upper_bound:.2f}): {len(high_outliers):,}')

In [ ]:
# Investigate zero and null prices
# Are they concentrated in a specific product, country or channel? 
# This can help identify if the issue is related to specific SKUs, regions, 
# or sales channels, which may point to data entry errors, promotional pricing, or other anomalies.

print('$0 Price Orders Investigation')
print(f'Total $0 price orders: {len(zero_prices):,}')
print()

print('Product drilldown:')
print(zero_prices['PRODUCT_ID'].value_counts().to_string())
print()

print('By marketing channel:')
print(zero_prices['MARKETING_CHANNEL'].value_counts().to_string())
print()

print('By purchase platform:')
print(zero_prices['PURCHASE_PLATFORM'].value_counts().to_string())
print()

print('By country(top 10):')
print(zero_prices['COUNTRY_CODE'].value_counts().head(10).to_string())
print()

In [ ]:
# Investigate high price outliers
# Are these real premium orders, or could they be data errors?

print('High Price Outliers Investigation')
print(f'Total high price outliers (> ${upper_bound:.2f}): {len(high_outliers):,}')
print()

print('Top 20 most expensive orders:')
top_prices = df.nlargest(20, 'USD_PRICE')[[
    'PRODUCT_NAME', 'USD_PRICE', 'COUNTRY_CODE', 
    'MARKETING_CHANNEL', 'PURCHASE_PLATFORM', 'PURCHASE_TS'
]]
print(top_prices.to_string(index=False))
print()

print('High price orders by product:')
print(high_outliers['PRODUCT_NAME'].value_counts().to_string())
print()

## Price Variation by Product

In [ ]:
# Calculate per-product price statistics to 
# identify if certain products have unusually high or low average prices, which could indicate pricing errors or misclassifications.

print('Per-product price statistics:')
product_price_stats = df.groupby('PRODUCT_NAME')['USD_PRICE'].agg([
    ('count',   'count'),
    ('mean',    'mean'),
    ('median',  'median'),
    ('std',     'std'),
    ('min',     'min'),
    ('max',     'max'),
    ('25%',     lambda x: x.quantile(0.25)),
    ('75%',     lambda x: x.quantile(0.75)),
]).round(2) # pyright: ignore[reportArgumentType, reportCallIssue]

# Calclulate IQR and outlier thresholds for each product

product_price_stats['IQR']         = product_price_stats['75%'] - product_price_stats['25%']
product_price_stats['lower_bound'] = product_price_stats['25%'] - (1.5 * product_price_stats['IQR'])
product_price_stats['upper_bound'] = product_price_stats['75%'] + (1.5 * product_price_stats['IQR'])
product_price_stats['price_range'] = product_price_stats['max'] - product_price_stats['min']

print(product_price_stats[['count', 'median', 'min', 'max', 'price_range', 'lower_bound', 'upper_bound']].to_string())

In [ ]:
# Flag per-product outliers using groupby and transform

# Calculate IQR and outlier thresholds for each product
# transform will return a series the same legth as df so we can create new columns to flag outliers

df['PRICE_Q1'] = df.groupby('PRODUCT_NAME')['USD_PRICE'].transform(lambda x: x.quantile(0.25))
df['PRICE_Q3'] = df.groupby('PRODUCT_NAME')['USD_PRICE'].transform(lambda x: x.quantile(0.75))
df['PRICE_IQR'] = df['PRICE_Q3'] - df['PRICE_Q1']

# Calculate lower and upper bounds for each product

df['PRICE_LOWER_BOUND'] = df['PRICE_Q1'] - (1.5 * df['PRICE_IQR'])
df['PRICE_UPPER_BOUND'] = df['PRICE_Q3'] + (1.5 * df['PRICE_IQR'])

# Flag outliers

df['IS_PRICE_OUTLIER'] = (
    (df['USD_PRICE'] < df['PRICE_LOWER_BOUND']) | 
    (df['USD_PRICE'] > df['PRICE_UPPER_BOUND'])
)

# Separatly flag $0 price orders

df['IS_ZERO_PRICE'] = df['USD_PRICE'] == 0

print('Per_Product outlier flags added:')
print(f" Total price outliers flagged:          {df['IS_PRICE_OUTLIER'].sum():,}")
print(f" Total $0 price orders flagged:         {df['IS_ZERO_PRICE'].sum():,}")
print()

print('Outlier count per product:')
outlier_by_product = df.groupby('PRODUCT_NAME')['IS_PRICE_OUTLIER'].agg(['sum', 'mean']).round(3)
outlier_by_product.columns = ['outlier_count', 'outlier_rate']
outlier_by_product['outlier_rate'] = (outlier_by_product['outlier_rate'] * 100).round(2) # Convert to percentage
print(outlier_by_product.sort_values('outlier_rate', ascending=False).to_string())